# Data Dharma by Srikanth
## SQL ↔ PySpark Bridge — Part 2
### Transformations: CASE WHEN, CAST, IN, LIKE, NULL Handling, String & Date Functions

### What will you learn?

Same dataset. Same rhythm as Part 1 — Business Requirement → SQL → Result → PySpark → Result → Mapping.

**Part 1 (already covered):**
SELECT → `select()`, WHERE → `filter()`, DISTINCT → `distinct()`, ORDER BY → `orderBy()`, LIMIT → `limit()`

**Part 2 (this notebook):**
CASE WHEN → `when().otherwise()`, CAST → `.cast()`, IN → `.isin()`, LIKE → `.like()`, NULL handling, String functions, Date functions

## SECTION 0 — Data Setup (extended for Part 2)

We're reusing the exact same `orders` dataset from Part 1 — same `order_id`, `order_date`, city, state, amounts. `order_amount` is calculated with the exact same formula, so nothing you already saw changes.

Three columns are added, each for a real teaching reason used later in this notebook:
- `customer_name` — some entries have extra spaces or inconsistent case, like real data, for the string-cleaning section
- `ship_date` — `PENDING` and `CANCELLED` orders genuinely have no ship date yet, so this is a real NULL, not a placeholder, for the NULL-handling and date sections
- `promo_code` — not every order used a promo code, for the COALESCE example

In [0]:
# from datetime is a module in Python and date/timedelta are classes in the datetime module
from datetime import date, timedelta

# from pyspark.sql.types is a module in Python and StructType is a class in it
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DateType,
    DoubleType,
)

# from pyspark.sql.functions is a module in Python
# each of these is a function in the pyspark.sql.functions module
from pyspark.sql.functions import (
    col,
    when,
    lit,
    coalesce,
    upper,
    lower,
    trim,
    length,
    concat,
    year,
    month,
    date_add,
    date_sub,
    datediff,
)

In [0]:
# below is the schema for the orders table
orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("customer_name", StringType(), False),
    StructField("order_date", DateType(), False),
    StructField("ship_date", DateType(), True),
    StructField("city", StringType(), False),
    StructField("state", StringType(), False),
    StructField("order_status", StringType(), False),
    StructField("payment_method", StringType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("unit_price", DoubleType(), False),
    StructField("discount_amount", DoubleType(), False),
    StructField("promo_code", StringType(), True),
])

In [0]:
# below is the list with sample data
orders_data = [
    (1001, 501, "  raj kumar",      date(2026, 1, 5),  date(2026, 1, 8),  "Houston",  "TX", "COMPLETED", "CREDIT_CARD", 3, 250.00, 20.00, "SAVE10"),
    (1002, 502, "MEENA REDDY ",     date(2026, 1, 6),  date(2026, 1, 8),  "Dallas",   "TX", "COMPLETED", "DEBIT_CARD",  2, 150.00, 10.00, None),
    (1003, 503, "Suresh Babu",      date(2026, 1, 7),  None,              "Austin",   "TX", "PENDING",   "PAYPAL",      1, 500.00,  0.00, "WELCOME5"),
    (1004, 504, "Anita Rao",        date(2026, 1, 8),  date(2026, 1, 12), "Chicago",  "IL", "COMPLETED", "UPI",         5,  80.00, 15.00, None),
    (1005, 505, "Kiran Varma",      date(2026, 1, 9),  None,              "New York", "NY", "CANCELLED", "CREDIT_CARD", 4, 120.00,  0.00, None),
    (1006, 501, " Raj Kumar",       date(2026, 1, 10), date(2026, 1, 13), "Houston",  "TX", "COMPLETED", "CREDIT_CARD", 2, 300.00, 25.00, "SAVE10"),
    (1007, 506, "Divya Nair",       date(2026, 1, 11), date(2026, 1, 13), "Dallas",   "TX", "RETURNED",  "DEBIT_CARD",  1, 200.00,  0.00, None),
    (1008, 507, "ramesh iyer",      date(2026, 1, 12), date(2026, 1, 17), "Austin",   "TX", "COMPLETED", "PAYPAL",      3, 100.00, 10.00, "FESTIVE20"),
    (1009, 508, "Priya Sharma",     date(2026, 1, 13), None,              "Chicago",  "IL", "PENDING",   "UPI",         2,  60.00,  5.00, None),
    (1010, 509, "Arjun Menon",      date(2026, 1, 14), date(2026, 1, 17), "New York", "NY", "COMPLETED", "CREDIT_CARD", 6,  90.00, 20.00, "WELCOME5"),
    (1011, 502, "MEENA REDDY",      date(2026, 1, 15), date(2026, 1, 17), "Dallas",   "TX", "COMPLETED", "DEBIT_CARD",  4, 175.00, 30.00, None),
    (1012, 510, "Lakshmi Pillai",   date(2026, 1, 16), date(2026, 1, 22), "Houston",  "TX", "COMPLETED", "PAYPAL",      1, 800.00, 50.00, "SAVE10"),
    (1013, 511, "Vikram Rao",       date(2026, 1, 17), None,              "Austin",   "TX", "CANCELLED", "UPI",         2, 250.00,  0.00, None),
    (1014, 512, "Sneha Gupta",      date(2026, 1, 18), date(2026, 1, 22), "Chicago",  "IL", "COMPLETED", "CREDIT_CARD", 3, 220.00, 10.00, "FESTIVE20"),
    (1015, 513, "Karthik Reddy",    date(2026, 1, 19), date(2026, 1, 22), "New York", "NY", "RETURNED",  "DEBIT_CARD",  1, 400.00,  0.00, None),
    (1016, 503, "  Suresh Babu  ",  date(2026, 1, 20), date(2026, 1, 22), "Austin",   "TX", "COMPLETED", "PAYPAL",      5,  60.00,  5.00, "WELCOME5"),
    (1017, 514, "Deepa Krishnan",   date(2026, 1, 21), None,              "Houston",  "TX", "PENDING",   "CREDIT_CARD", 2, 175.00,  0.00, None),
    (1018, 505, "Kiran Varma",      date(2026, 1, 22), date(2026, 1, 25), "New York", "NY", "COMPLETED", "UPI",         3, 210.00, 15.00, "SAVE10"),
]

# create a dataframe with the schema and data
orders_raw_df = spark.createDataFrame(orders_data, schema=orders_schema)

# display the dataframe
display(orders_raw_df)

order_id,customer_id,customer_name,order_date,ship_date,city,state,order_status,payment_method,quantity,unit_price,discount_amount,promo_code
1001,501,raj kumar,2026-01-05,2026-01-08,Houston,TX,COMPLETED,CREDIT_CARD,3,250.0,20.0,SAVE10
1002,502,MEENA REDDY,2026-01-06,2026-01-08,Dallas,TX,COMPLETED,DEBIT_CARD,2,150.0,10.0,null
1003,503,Suresh Babu,2026-01-07,null,Austin,TX,PENDING,PAYPAL,1,500.0,0.0,WELCOME5
1004,504,Anita Rao,2026-01-08,2026-01-12,Chicago,IL,COMPLETED,UPI,5,80.0,15.0,null
1005,505,Kiran Varma,2026-01-09,null,New York,NY,CANCELLED,CREDIT_CARD,4,120.0,0.0,null
1006,501,Raj Kumar,2026-01-10,2026-01-13,Houston,TX,COMPLETED,CREDIT_CARD,2,300.0,25.0,SAVE10
1007,506,Divya Nair,2026-01-11,2026-01-13,Dallas,TX,RETURNED,DEBIT_CARD,1,200.0,0.0,null
1008,507,ramesh iyer,2026-01-12,2026-01-17,Austin,TX,COMPLETED,PAYPAL,3,100.0,10.0,FESTIVE20
1009,508,Priya Sharma,2026-01-13,null,Chicago,IL,PENDING,UPI,2,60.0,5.0,null
1010,509,Arjun Menon,2026-01-14,2026-01-17,New York,NY,COMPLETED,CREDIT_CARD,6,90.0,20.0,WELCOME5


**Derived business measure (same formula as Part 1):**

```
order_amount = (quantity * unit_price) - discount_amount
```

In [0]:
# create a new column 'order_amount' by multiplying 'quantity' and 'unit_price' and subtracting 'discount_amount'
# round the result to 2 decimal places and cast it to decimal(10,2)
from pyspark.sql.functions import round as spark_round

orders_df = orders_raw_df.withColumn(
    "order_amount",
    spark_round((col("quantity") * col("unit_price")) - col("discount_amount"), 2).cast("decimal(10,2)")
)

# display the dataframe
display(orders_df)

order_id,customer_id,customer_name,order_date,ship_date,city,state,order_status,payment_method,quantity,unit_price,discount_amount,promo_code,order_amount
1001,501,raj kumar,2026-01-05,2026-01-08,Houston,TX,COMPLETED,CREDIT_CARD,3,250.0,20.0,SAVE10,730.00
1002,502,MEENA REDDY,2026-01-06,2026-01-08,Dallas,TX,COMPLETED,DEBIT_CARD,2,150.0,10.0,null,290.00
1003,503,Suresh Babu,2026-01-07,null,Austin,TX,PENDING,PAYPAL,1,500.0,0.0,WELCOME5,500.00
1004,504,Anita Rao,2026-01-08,2026-01-12,Chicago,IL,COMPLETED,UPI,5,80.0,15.0,null,385.00
1005,505,Kiran Varma,2026-01-09,null,New York,NY,CANCELLED,CREDIT_CARD,4,120.0,0.0,null,480.00
1006,501,Raj Kumar,2026-01-10,2026-01-13,Houston,TX,COMPLETED,CREDIT_CARD,2,300.0,25.0,SAVE10,575.00
1007,506,Divya Nair,2026-01-11,2026-01-13,Dallas,TX,RETURNED,DEBIT_CARD,1,200.0,0.0,null,200.00
1008,507,ramesh iyer,2026-01-12,2026-01-17,Austin,TX,COMPLETED,PAYPAL,3,100.0,10.0,FESTIVE20,290.00
1009,508,Priya Sharma,2026-01-13,null,Chicago,IL,PENDING,UPI,2,60.0,5.0,null,115.00
1010,509,Arjun Menon,2026-01-14,2026-01-17,New York,NY,COMPLETED,CREDIT_CARD,6,90.0,20.0,WELCOME5,520.00


In [0]:
# create a temp view 'orders' from the dataframe
orders_df.createOrReplaceTempView("orders")

From this point onward, SQL and PySpark are reading the same data —
SQL through the view `orders`, PySpark through the DataFrame `orders_df`.

## SECTION 1 — CASE WHEN

**Business Requirement:** Categorize each order as `HIGH`, `MEDIUM`, or `LOW` based on `order_amount`.

### 🟨 SQL

In [0]:
%sql
-- Categorize orders as HIGH, MEDIUM, or LOW based on order_amount

SELECT
    order_id,
    order_amount,
    CASE
        WHEN order_amount >= 500 THEN 'HIGH'
        WHEN order_amount >= 200 THEN 'MEDIUM'
        ELSE 'LOW'
    END AS order_category
FROM orders;

order_id,order_amount,order_category
1001,730.00,HIGH
1002,290.00,MEDIUM
1003,500.00,HIGH
1004,385.00,MEDIUM
1005,480.00,MEDIUM
1006,575.00,HIGH
1007,200.00,MEDIUM
1008,290.00,MEDIUM
1009,115.00,LOW
1010,520.00,HIGH


### 🟦 PySpark

In [0]:
# Categorize orders as HIGH, MEDIUM, or LOW based on order_amount

result_df = orders_df.select(
    "order_id",
    "order_amount",
    when(col("order_amount") >= 500, "HIGH")
    .when(col("order_amount") >= 200, "MEDIUM")
    .otherwise("LOW")
    .alias("order_category")
)

# display data in the DataFrame result_df
display(result_df)

order_id,order_amount,order_category
1001,730.00,HIGH
1002,290.00,MEDIUM
1003,500.00,HIGH
1004,385.00,MEDIUM
1005,480.00,MEDIUM
1006,575.00,HIGH
1007,200.00,MEDIUM
1008,290.00,MEDIUM
1009,115.00,LOW
1010,520.00,HIGH


**Key Mapping**

| SQL | PySpark |
|---|---|
| `CASE WHEN ... THEN ...` | `when(condition, value)` |
| additional `WHEN` | `.when(condition, value)` |
| `ELSE` | `.otherwise(value)` |
| `AS order_category` | `.alias("order_category")` |

`CASE WHEN` ↔ `when().otherwise()`

## SECTION 2 — CAST

**Business Requirement:** Convert `order_id` to a text value, so it can be combined with labels later (we'll reuse this in Section 6).

### 🟨 SQL

In [0]:
%sql
-- Convert order_id to a text value

SELECT
    order_id,
    CAST(order_id AS STRING) AS order_id_text
FROM orders;

order_id,order_id_text
1001,1001
1002,1002
1003,1003
1004,1004
1005,1005
1006,1006
1007,1007
1008,1008
1009,1009
1010,1010


### 🟦 PySpark

In [0]:
# Convert order_id to a text value

result_df = orders_df.select(
    "order_id",
    col("order_id").cast("string").alias("order_id_text")
)

# display data in the DataFrame result_df
display(result_df)

order_id,order_id_text
1001,1001
1002,1002
1003,1003
1004,1004
1005,1005
1006,1006
1007,1007
1008,1008
1009,1009
1010,1010


**Key Mapping**

`CAST(column AS type)` ↔ `col("column").cast("type")`

## SECTION 3 — IN

**Business Requirement:** Return orders from Houston, Dallas, or Austin.

### 🟨 SQL

In [0]:
%sql
-- Return orders from Houston, Dallas, or Austin

SELECT *
FROM orders
WHERE city IN ('Houston', 'Dallas', 'Austin');

order_id,customer_id,customer_name,order_date,ship_date,city,state,order_status,payment_method,quantity,unit_price,discount_amount,promo_code,order_amount
1001,501,raj kumar,2026-01-05,2026-01-08,Houston,TX,COMPLETED,CREDIT_CARD,3,250.0,20.0,SAVE10,730.00
1002,502,MEENA REDDY,2026-01-06,2026-01-08,Dallas,TX,COMPLETED,DEBIT_CARD,2,150.0,10.0,null,290.00
1003,503,Suresh Babu,2026-01-07,null,Austin,TX,PENDING,PAYPAL,1,500.0,0.0,WELCOME5,500.00
1006,501,Raj Kumar,2026-01-10,2026-01-13,Houston,TX,COMPLETED,CREDIT_CARD,2,300.0,25.0,SAVE10,575.00
1007,506,Divya Nair,2026-01-11,2026-01-13,Dallas,TX,RETURNED,DEBIT_CARD,1,200.0,0.0,null,200.00
1008,507,ramesh iyer,2026-01-12,2026-01-17,Austin,TX,COMPLETED,PAYPAL,3,100.0,10.0,FESTIVE20,290.00
1011,502,MEENA REDDY,2026-01-15,2026-01-17,Dallas,TX,COMPLETED,DEBIT_CARD,4,175.0,30.0,null,670.00
1012,510,Lakshmi Pillai,2026-01-16,2026-01-22,Houston,TX,COMPLETED,PAYPAL,1,800.0,50.0,SAVE10,750.00
1013,511,Vikram Rao,2026-01-17,null,Austin,TX,CANCELLED,UPI,2,250.0,0.0,null,500.00
1016,503,Suresh Babu,2026-01-20,2026-01-22,Austin,TX,COMPLETED,PAYPAL,5,60.0,5.0,WELCOME5,295.00


### 🟦 PySpark

In [0]:
# Return orders from Houston, Dallas, or Austin

result_df = orders_df.filter(
    col("city").isin("Houston", "Dallas", "Austin")
)

# display data in the DataFrame result_df
display(result_df)

order_id,customer_id,customer_name,order_date,ship_date,city,state,order_status,payment_method,quantity,unit_price,discount_amount,promo_code,order_amount
1001,501,raj kumar,2026-01-05,2026-01-08,Houston,TX,COMPLETED,CREDIT_CARD,3,250.0,20.0,SAVE10,730.00
1002,502,MEENA REDDY,2026-01-06,2026-01-08,Dallas,TX,COMPLETED,DEBIT_CARD,2,150.0,10.0,null,290.00
1003,503,Suresh Babu,2026-01-07,null,Austin,TX,PENDING,PAYPAL,1,500.0,0.0,WELCOME5,500.00
1006,501,Raj Kumar,2026-01-10,2026-01-13,Houston,TX,COMPLETED,CREDIT_CARD,2,300.0,25.0,SAVE10,575.00
1007,506,Divya Nair,2026-01-11,2026-01-13,Dallas,TX,RETURNED,DEBIT_CARD,1,200.0,0.0,null,200.00
1008,507,ramesh iyer,2026-01-12,2026-01-17,Austin,TX,COMPLETED,PAYPAL,3,100.0,10.0,FESTIVE20,290.00
1011,502,MEENA REDDY,2026-01-15,2026-01-17,Dallas,TX,COMPLETED,DEBIT_CARD,4,175.0,30.0,null,670.00
1012,510,Lakshmi Pillai,2026-01-16,2026-01-22,Houston,TX,COMPLETED,PAYPAL,1,800.0,50.0,SAVE10,750.00
1013,511,Vikram Rao,2026-01-17,null,Austin,TX,CANCELLED,UPI,2,250.0,0.0,null,500.00
1016,503,Suresh Babu,2026-01-20,2026-01-22,Austin,TX,COMPLETED,PAYPAL,5,60.0,5.0,WELCOME5,295.00


**Key Mapping**

`IN (...)` ↔ `.isin(...)`

## SECTION 4 — LIKE

**Business Requirement:** Find orders from cities starting with the letter H.

`%` matches any number of characters. `_` matches exactly one character.

### 🟨 SQL

In [0]:
%sql
-- Find orders from cities starting with H

SELECT *
FROM orders
WHERE city LIKE 'H%';

order_id,customer_id,customer_name,order_date,ship_date,city,state,order_status,payment_method,quantity,unit_price,discount_amount,promo_code,order_amount
1001,501,raj kumar,2026-01-05,2026-01-08,Houston,TX,COMPLETED,CREDIT_CARD,3,250.0,20.0,SAVE10,730.00
1006,501,Raj Kumar,2026-01-10,2026-01-13,Houston,TX,COMPLETED,CREDIT_CARD,2,300.0,25.0,SAVE10,575.00
1012,510,Lakshmi Pillai,2026-01-16,2026-01-22,Houston,TX,COMPLETED,PAYPAL,1,800.0,50.0,SAVE10,750.00
1017,514,Deepa Krishnan,2026-01-21,null,Houston,TX,PENDING,CREDIT_CARD,2,175.0,0.0,null,350.00


### 🟦 PySpark

In [0]:
# Find orders from cities starting with H

result_df = orders_df.filter(
    col("city").like("H%")
)

# display data in the DataFrame result_df
display(result_df)

order_id,customer_id,customer_name,order_date,ship_date,city,state,order_status,payment_method,quantity,unit_price,discount_amount,promo_code,order_amount
1001,501,raj kumar,2026-01-05,2026-01-08,Houston,TX,COMPLETED,CREDIT_CARD,3,250.0,20.0,SAVE10,730.00
1006,501,Raj Kumar,2026-01-10,2026-01-13,Houston,TX,COMPLETED,CREDIT_CARD,2,300.0,25.0,SAVE10,575.00
1012,510,Lakshmi Pillai,2026-01-16,2026-01-22,Houston,TX,COMPLETED,PAYPAL,1,800.0,50.0,SAVE10,750.00
1017,514,Deepa Krishnan,2026-01-21,null,Houston,TX,PENDING,CREDIT_CARD,2,175.0,0.0,null,350.00


**One more example:** Find orders paid using any type of card.

### 🟨 SQL

In [0]:
%sql
-- Find orders paid using any type of card

SELECT *
FROM orders
WHERE payment_method LIKE '%CARD';

order_id,customer_id,customer_name,order_date,ship_date,city,state,order_status,payment_method,quantity,unit_price,discount_amount,promo_code,order_amount
1001,501,raj kumar,2026-01-05,2026-01-08,Houston,TX,COMPLETED,CREDIT_CARD,3,250.0,20.0,SAVE10,730.00
1002,502,MEENA REDDY,2026-01-06,2026-01-08,Dallas,TX,COMPLETED,DEBIT_CARD,2,150.0,10.0,null,290.00
1005,505,Kiran Varma,2026-01-09,null,New York,NY,CANCELLED,CREDIT_CARD,4,120.0,0.0,null,480.00
1006,501,Raj Kumar,2026-01-10,2026-01-13,Houston,TX,COMPLETED,CREDIT_CARD,2,300.0,25.0,SAVE10,575.00
1007,506,Divya Nair,2026-01-11,2026-01-13,Dallas,TX,RETURNED,DEBIT_CARD,1,200.0,0.0,null,200.00
1010,509,Arjun Menon,2026-01-14,2026-01-17,New York,NY,COMPLETED,CREDIT_CARD,6,90.0,20.0,WELCOME5,520.00
1011,502,MEENA REDDY,2026-01-15,2026-01-17,Dallas,TX,COMPLETED,DEBIT_CARD,4,175.0,30.0,null,670.00
1014,512,Sneha Gupta,2026-01-18,2026-01-22,Chicago,IL,COMPLETED,CREDIT_CARD,3,220.0,10.0,FESTIVE20,650.00
1015,513,Karthik Reddy,2026-01-19,2026-01-22,New York,NY,RETURNED,DEBIT_CARD,1,400.0,0.0,null,400.00
1017,514,Deepa Krishnan,2026-01-21,null,Houston,TX,PENDING,CREDIT_CARD,2,175.0,0.0,null,350.00


### 🟦 PySpark

In [0]:
# Find orders paid using any type of card

result_df = orders_df.filter(
    col("payment_method").like("%CARD")
)

# display data in the DataFrame result_df
display(result_df)

order_id,customer_id,customer_name,order_date,ship_date,city,state,order_status,payment_method,quantity,unit_price,discount_amount,promo_code,order_amount
1001,501,raj kumar,2026-01-05,2026-01-08,Houston,TX,COMPLETED,CREDIT_CARD,3,250.0,20.0,SAVE10,730.00
1002,502,MEENA REDDY,2026-01-06,2026-01-08,Dallas,TX,COMPLETED,DEBIT_CARD,2,150.0,10.0,null,290.00
1005,505,Kiran Varma,2026-01-09,null,New York,NY,CANCELLED,CREDIT_CARD,4,120.0,0.0,null,480.00
1006,501,Raj Kumar,2026-01-10,2026-01-13,Houston,TX,COMPLETED,CREDIT_CARD,2,300.0,25.0,SAVE10,575.00
1007,506,Divya Nair,2026-01-11,2026-01-13,Dallas,TX,RETURNED,DEBIT_CARD,1,200.0,0.0,null,200.00
1010,509,Arjun Menon,2026-01-14,2026-01-17,New York,NY,COMPLETED,CREDIT_CARD,6,90.0,20.0,WELCOME5,520.00
1011,502,MEENA REDDY,2026-01-15,2026-01-17,Dallas,TX,COMPLETED,DEBIT_CARD,4,175.0,30.0,null,670.00
1014,512,Sneha Gupta,2026-01-18,2026-01-22,Chicago,IL,COMPLETED,CREDIT_CARD,3,220.0,10.0,FESTIVE20,650.00
1015,513,Karthik Reddy,2026-01-19,2026-01-22,New York,NY,RETURNED,DEBIT_CARD,1,400.0,0.0,null,400.00
1017,514,Deepa Krishnan,2026-01-21,null,Houston,TX,PENDING,CREDIT_CARD,2,175.0,0.0,null,350.00


**Key Mapping**

`LIKE` ↔ `.like()`

## SECTION 5 — NULL HANDLING

**Important:** `NULL` does not mean `0`, an empty string, or `False`. `NULL` means the value is missing or unknown.

`ship_date` is `NULL` for orders that are still `PENDING` or were `CANCELLED` — they were genuinely never shipped, so there's no ship date to store. `promo_code` is `NULL` for orders where no promo code was used.

**A. IS NULL** — Business Requirement: Find orders that have not been shipped yet.

### 🟨 SQL

In [0]:
%sql
-- Find orders that have not been shipped yet

SELECT *
FROM orders
WHERE ship_date IS NULL;

order_id,customer_id,customer_name,order_date,ship_date,city,state,order_status,payment_method,quantity,unit_price,discount_amount,promo_code,order_amount
1003,503,Suresh Babu,2026-01-07,null,Austin,TX,PENDING,PAYPAL,1,500.0,0.0,WELCOME5,500.00
1005,505,Kiran Varma,2026-01-09,null,New York,NY,CANCELLED,CREDIT_CARD,4,120.0,0.0,null,480.00
1009,508,Priya Sharma,2026-01-13,null,Chicago,IL,PENDING,UPI,2,60.0,5.0,null,115.00
1013,511,Vikram Rao,2026-01-17,null,Austin,TX,CANCELLED,UPI,2,250.0,0.0,null,500.00
1017,514,Deepa Krishnan,2026-01-21,null,Houston,TX,PENDING,CREDIT_CARD,2,175.0,0.0,null,350.00


### 🟦 PySpark

In [0]:
# Find orders that have not been shipped yet

result_df = orders_df.filter(
    col("ship_date").isNull()
)

# display data in the DataFrame result_df
display(result_df)

order_id,customer_id,customer_name,order_date,ship_date,city,state,order_status,payment_method,quantity,unit_price,discount_amount,promo_code,order_amount
1003,503,Suresh Babu,2026-01-07,null,Austin,TX,PENDING,PAYPAL,1,500.0,0.0,WELCOME5,500.00
1005,505,Kiran Varma,2026-01-09,null,New York,NY,CANCELLED,CREDIT_CARD,4,120.0,0.0,null,480.00
1009,508,Priya Sharma,2026-01-13,null,Chicago,IL,PENDING,UPI,2,60.0,5.0,null,115.00
1013,511,Vikram Rao,2026-01-17,null,Austin,TX,CANCELLED,UPI,2,250.0,0.0,null,500.00
1017,514,Deepa Krishnan,2026-01-21,null,Houston,TX,PENDING,CREDIT_CARD,2,175.0,0.0,null,350.00


**B. IS NOT NULL** — Business Requirement: Find orders that have already been shipped.

### 🟨 SQL

In [0]:
%sql
-- Find orders that have already been shipped

SELECT *
FROM orders
WHERE ship_date IS NOT NULL;

order_id,customer_id,customer_name,order_date,ship_date,city,state,order_status,payment_method,quantity,unit_price,discount_amount,promo_code,order_amount
1001,501,raj kumar,2026-01-05,2026-01-08,Houston,TX,COMPLETED,CREDIT_CARD,3,250.0,20.0,SAVE10,730.00
1002,502,MEENA REDDY,2026-01-06,2026-01-08,Dallas,TX,COMPLETED,DEBIT_CARD,2,150.0,10.0,null,290.00
1004,504,Anita Rao,2026-01-08,2026-01-12,Chicago,IL,COMPLETED,UPI,5,80.0,15.0,null,385.00
1006,501,Raj Kumar,2026-01-10,2026-01-13,Houston,TX,COMPLETED,CREDIT_CARD,2,300.0,25.0,SAVE10,575.00
1007,506,Divya Nair,2026-01-11,2026-01-13,Dallas,TX,RETURNED,DEBIT_CARD,1,200.0,0.0,null,200.00
1008,507,ramesh iyer,2026-01-12,2026-01-17,Austin,TX,COMPLETED,PAYPAL,3,100.0,10.0,FESTIVE20,290.00
1010,509,Arjun Menon,2026-01-14,2026-01-17,New York,NY,COMPLETED,CREDIT_CARD,6,90.0,20.0,WELCOME5,520.00
1011,502,MEENA REDDY,2026-01-15,2026-01-17,Dallas,TX,COMPLETED,DEBIT_CARD,4,175.0,30.0,null,670.00
1012,510,Lakshmi Pillai,2026-01-16,2026-01-22,Houston,TX,COMPLETED,PAYPAL,1,800.0,50.0,SAVE10,750.00
1014,512,Sneha Gupta,2026-01-18,2026-01-22,Chicago,IL,COMPLETED,CREDIT_CARD,3,220.0,10.0,FESTIVE20,650.00


### 🟦 PySpark

In [0]:
# Find orders that have already been shipped

result_df = orders_df.filter(
    col("ship_date").isNotNull()
)

# display data in the DataFrame result_df
display(result_df)

order_id,customer_id,customer_name,order_date,ship_date,city,state,order_status,payment_method,quantity,unit_price,discount_amount,promo_code,order_amount
1001,501,raj kumar,2026-01-05,2026-01-08,Houston,TX,COMPLETED,CREDIT_CARD,3,250.0,20.0,SAVE10,730.00
1002,502,MEENA REDDY,2026-01-06,2026-01-08,Dallas,TX,COMPLETED,DEBIT_CARD,2,150.0,10.0,null,290.00
1004,504,Anita Rao,2026-01-08,2026-01-12,Chicago,IL,COMPLETED,UPI,5,80.0,15.0,null,385.00
1006,501,Raj Kumar,2026-01-10,2026-01-13,Houston,TX,COMPLETED,CREDIT_CARD,2,300.0,25.0,SAVE10,575.00
1007,506,Divya Nair,2026-01-11,2026-01-13,Dallas,TX,RETURNED,DEBIT_CARD,1,200.0,0.0,null,200.00
1008,507,ramesh iyer,2026-01-12,2026-01-17,Austin,TX,COMPLETED,PAYPAL,3,100.0,10.0,FESTIVE20,290.00
1010,509,Arjun Menon,2026-01-14,2026-01-17,New York,NY,COMPLETED,CREDIT_CARD,6,90.0,20.0,WELCOME5,520.00
1011,502,MEENA REDDY,2026-01-15,2026-01-17,Dallas,TX,COMPLETED,DEBIT_CARD,4,175.0,30.0,null,670.00
1012,510,Lakshmi Pillai,2026-01-16,2026-01-22,Houston,TX,COMPLETED,PAYPAL,1,800.0,50.0,SAVE10,750.00
1014,512,Sneha Gupta,2026-01-18,2026-01-22,Chicago,IL,COMPLETED,CREDIT_CARD,3,220.0,10.0,FESTIVE20,650.00


**C. COALESCE** — Business Requirement: Replace missing promo codes with 'NONE' for reporting.

### 🟨 SQL

In [0]:
%sql
-- Replace missing promo codes with 'NONE'

SELECT
    order_id,
    promo_code,
    COALESCE(promo_code, 'NONE') AS promo_code_clean
FROM orders;

order_id,promo_code,promo_code_clean
1001,SAVE10,SAVE10
1002,null,NONE
1003,WELCOME5,WELCOME5
1004,null,NONE
1005,null,NONE
1006,SAVE10,SAVE10
1007,null,NONE
1008,FESTIVE20,FESTIVE20
1009,null,NONE
1010,WELCOME5,WELCOME5


### 🟦 PySpark

In [0]:
# Replace missing promo codes with 'NONE'

result_df = orders_df.select(
    "order_id",
    "promo_code",
    coalesce(col("promo_code"), lit("NONE")).alias("promo_code_clean")
)

# display data in the DataFrame result_df
display(result_df)

order_id,promo_code,promo_code_clean
1001,SAVE10,SAVE10
1002,null,NONE
1003,WELCOME5,WELCOME5
1004,null,NONE
1005,null,NONE
1006,SAVE10,SAVE10
1007,null,NONE
1008,FESTIVE20,FESTIVE20
1009,null,NONE
1010,WELCOME5,WELCOME5


**Key Mapping**

| SQL | PySpark |
|---|---|
| `IS NULL` | `.isNull()` |
| `IS NOT NULL` | `.isNotNull()` |
| `COALESCE(column, default)` | `coalesce(col("column"), lit(default))` |

## SECTION 6 — STRING FUNCTIONS

**Business Requirement:** Clean up `customer_name` for consistent display — remove extra spaces, standardize capitalization, and check name length as a data-quality check.

### 🟨 SQL

In [0]:
%sql
-- Clean up customer_name by trimming spaces, standardizing case, and checking length

SELECT
    order_id,
    customer_name,
    TRIM(customer_name) AS trimmed_name,
    UPPER(TRIM(customer_name)) AS upper_name,
    LOWER(TRIM(customer_name)) AS lower_name,
    LENGTH(TRIM(customer_name)) AS name_length
FROM orders;

order_id,customer_name,trimmed_name,upper_name,lower_name,name_length
1001,raj kumar,raj kumar,RAJ KUMAR,raj kumar,9
1002,MEENA REDDY,MEENA REDDY,MEENA REDDY,meena reddy,11
1003,Suresh Babu,Suresh Babu,SURESH BABU,suresh babu,11
1004,Anita Rao,Anita Rao,ANITA RAO,anita rao,9
1005,Kiran Varma,Kiran Varma,KIRAN VARMA,kiran varma,11
1006,Raj Kumar,Raj Kumar,RAJ KUMAR,raj kumar,9
1007,Divya Nair,Divya Nair,DIVYA NAIR,divya nair,10
1008,ramesh iyer,ramesh iyer,RAMESH IYER,ramesh iyer,11
1009,Priya Sharma,Priya Sharma,PRIYA SHARMA,priya sharma,12
1010,Arjun Menon,Arjun Menon,ARJUN MENON,arjun menon,11


### 🟦 PySpark

In [0]:
# Clean up customer_name by trimming spaces, standardizing case, and checking length

result_df = orders_df.select(
    "order_id",
    "customer_name",
    trim(col("customer_name")).alias("trimmed_name"),
    upper(trim(col("customer_name"))).alias("upper_name"),
    lower(trim(col("customer_name"))).alias("lower_name"),
    length(trim(col("customer_name"))).alias("name_length")
)

# display data in the DataFrame result_df
display(result_df)

order_id,customer_name,trimmed_name,upper_name,lower_name,name_length
1001,raj kumar,raj kumar,RAJ KUMAR,raj kumar,9
1002,MEENA REDDY,MEENA REDDY,MEENA REDDY,meena reddy,11
1003,Suresh Babu,Suresh Babu,SURESH BABU,suresh babu,11
1004,Anita Rao,Anita Rao,ANITA RAO,anita rao,9
1005,Kiran Varma,Kiran Varma,KIRAN VARMA,kiran varma,11
1006,Raj Kumar,Raj Kumar,RAJ KUMAR,raj kumar,9
1007,Divya Nair,Divya Nair,DIVYA NAIR,divya nair,10
1008,ramesh iyer,ramesh iyer,RAMESH IYER,ramesh iyer,11
1009,Priya Sharma,Priya Sharma,PRIYA SHARMA,priya sharma,12
1010,Arjun Menon,Arjun Menon,ARJUN MENON,arjun menon,11


**One more example:** Build a friendly order label combining the order reference and city.

### 🟨 SQL

In [0]:
%sql
-- Build a friendly order label combining the order reference and city

SELECT
    order_id,
    CONCAT('ORD-', CAST(order_id AS STRING), ' - ', city) AS order_label
FROM orders;

order_id,order_label
1001,ORD-1001 - Houston
1002,ORD-1002 - Dallas
1003,ORD-1003 - Austin
1004,ORD-1004 - Chicago
1005,ORD-1005 - New York
1006,ORD-1006 - Houston
1007,ORD-1007 - Dallas
1008,ORD-1008 - Austin
1009,ORD-1009 - Chicago
1010,ORD-1010 - New York


### 🟦 PySpark

In [0]:
# Build a friendly order label combining the order reference and city

result_df = orders_df.select(
    "order_id",
    concat(lit("ORD-"), col("order_id").cast("string"), lit(" - "), col("city")).alias("order_label")
)

# display data in the DataFrame result_df
display(result_df)

order_id,order_label
1001,ORD-1001 - Houston
1002,ORD-1002 - Dallas
1003,ORD-1003 - Austin
1004,ORD-1004 - Chicago
1005,ORD-1005 - New York
1006,ORD-1006 - Houston
1007,ORD-1007 - Dallas
1008,ORD-1008 - Austin
1009,ORD-1009 - Chicago
1010,ORD-1010 - New York


**Key Mapping**

| SQL | PySpark |
|---|---|
| `UPPER` | `upper()` |
| `LOWER` | `lower()` |
| `TRIM` | `trim()` |
| `LENGTH` | `length()` |
| `CONCAT` | `concat()` |

## SECTION 7 — DATE FUNCTIONS

**Business Requirement:** Find the year and month each order was placed.

### 🟨 SQL

In [0]:
%sql
-- Find the year and month each order was placed

SELECT
    order_id,
    order_date,
    YEAR(order_date) AS order_year,
    MONTH(order_date) AS order_month
FROM orders;

order_id,order_date,order_year,order_month
1001,2026-01-05,2026,1
1002,2026-01-06,2026,1
1003,2026-01-07,2026,1
1004,2026-01-08,2026,1
1005,2026-01-09,2026,1
1006,2026-01-10,2026,1
1007,2026-01-11,2026,1
1008,2026-01-12,2026,1
1009,2026-01-13,2026,1
1010,2026-01-14,2026,1


### 🟦 PySpark

In [0]:
# Find the year and month each order was placed

result_df = orders_df.select(
    "order_id",
    "order_date",
    year(col("order_date")).alias("order_year"),
    month(col("order_date")).alias("order_month")
)

# display data in the DataFrame result_df
display(result_df)

order_id,order_date,order_year,order_month
1001,2026-01-05,2026,1
1002,2026-01-06,2026,1
1003,2026-01-07,2026,1
1004,2026-01-08,2026,1
1005,2026-01-09,2026,1
1006,2026-01-10,2026,1
1007,2026-01-11,2026,1
1008,2026-01-12,2026,1
1009,2026-01-13,2026,1
1010,2026-01-14,2026,1


**Next:** Calculate the promised delivery date, 7 days after the order date.

### 🟨 SQL

In [0]:
%sql
-- Calculate the promised delivery date, 7 days after the order date

SELECT
    order_id,
    order_date,
    DATE_ADD(order_date, 7) AS promised_delivery_date
FROM orders;

order_id,order_date,promised_delivery_date
1001,2026-01-05,2026-01-12
1002,2026-01-06,2026-01-13
1003,2026-01-07,2026-01-14
1004,2026-01-08,2026-01-15
1005,2026-01-09,2026-01-16
1006,2026-01-10,2026-01-17
1007,2026-01-11,2026-01-18
1008,2026-01-12,2026-01-19
1009,2026-01-13,2026-01-20
1010,2026-01-14,2026-01-21


### 🟦 PySpark

In [0]:
# Calculate the promised delivery date, 7 days after the order date

result_df = orders_df.select(
    "order_id",
    "order_date",
    date_add(col("order_date"), 7).alias("promised_delivery_date")
)

# display data in the DataFrame result_df
display(result_df)

order_id,order_date,promised_delivery_date
1001,2026-01-05,2026-01-12
1002,2026-01-06,2026-01-13
1003,2026-01-07,2026-01-14
1004,2026-01-08,2026-01-15
1005,2026-01-09,2026-01-16
1006,2026-01-10,2026-01-17
1007,2026-01-11,2026-01-18
1008,2026-01-12,2026-01-19
1009,2026-01-13,2026-01-20
1010,2026-01-14,2026-01-21


**Next:** Find the date one week before the order date, used for promo eligibility lookback.

### 🟨 SQL

In [0]:
%sql
-- Find the date one week before the order date, for promo eligibility lookback

SELECT
    order_id,
    order_date,
    DATE_SUB(order_date, 7) AS lookback_date
FROM orders;

order_id,order_date,lookback_date
1001,2026-01-05,2025-12-29
1002,2026-01-06,2025-12-30
1003,2026-01-07,2025-12-31
1004,2026-01-08,2026-01-01
1005,2026-01-09,2026-01-02
1006,2026-01-10,2026-01-03
1007,2026-01-11,2026-01-04
1008,2026-01-12,2026-01-05
1009,2026-01-13,2026-01-06
1010,2026-01-14,2026-01-07


### 🟦 PySpark

In [0]:
# Find the date one week before the order date, for promo eligibility lookback

result_df = orders_df.select(
    "order_id",
    "order_date",
    date_sub(col("order_date"), 7).alias("lookback_date")
)

# display data in the DataFrame result_df
display(result_df)

order_id,order_date,lookback_date
1001,2026-01-05,2025-12-29
1002,2026-01-06,2025-12-30
1003,2026-01-07,2025-12-31
1004,2026-01-08,2026-01-01
1005,2026-01-09,2026-01-02
1006,2026-01-10,2026-01-03
1007,2026-01-11,2026-01-04
1008,2026-01-12,2026-01-05
1009,2026-01-13,2026-01-06
1010,2026-01-14,2026-01-07


**Next:** Calculate how many days it took to ship each order that has shipped.

### 🟨 SQL

In [0]:
%sql
-- Calculate how many days it took to ship each order that has shipped

SELECT
    order_id,
    order_date,
    ship_date,
    DATEDIFF(ship_date, order_date) AS days_to_ship
FROM orders
WHERE ship_date IS NOT NULL;

order_id,order_date,ship_date,days_to_ship
1001,2026-01-05,2026-01-08,3
1002,2026-01-06,2026-01-08,2
1004,2026-01-08,2026-01-12,4
1006,2026-01-10,2026-01-13,3
1007,2026-01-11,2026-01-13,2
1008,2026-01-12,2026-01-17,5
1010,2026-01-14,2026-01-17,3
1011,2026-01-15,2026-01-17,2
1012,2026-01-16,2026-01-22,6
1014,2026-01-18,2026-01-22,4


### 🟦 PySpark

In [0]:
# Calculate how many days it took to ship each order that has shipped

result_df = (
    orders_df
    .filter(col("ship_date").isNotNull())
    .select(
        "order_id",
        "order_date",
        "ship_date",
        datediff(col("ship_date"), col("order_date")).alias("days_to_ship")
    )
)

# display data in the DataFrame result_df
display(result_df)

order_id,order_date,ship_date,days_to_ship
1001,2026-01-05,2026-01-08,3
1002,2026-01-06,2026-01-08,2
1004,2026-01-08,2026-01-12,4
1006,2026-01-10,2026-01-13,3
1007,2026-01-11,2026-01-13,2
1008,2026-01-12,2026-01-17,5
1010,2026-01-14,2026-01-17,3
1011,2026-01-15,2026-01-17,2
1012,2026-01-16,2026-01-22,6
1014,2026-01-18,2026-01-22,4


**Key Mapping**

| SQL | PySpark |
|---|---|
| `YEAR` | `year()` |
| `MONTH` | `month()` |
| `DATE_ADD` | `date_add()` |
| `DATE_SUB` | `date_sub()` |
| `DATEDIFF` | `datediff()` |

## SECTION 8 — COMBINING CONCEPTS

**Business Requirement:** For completed Texas orders — clean the customer name, categorize by order amount, replace missing promo codes with 'NONE', and calculate days to ship.

### 🟨 SQL

In [0]:
%sql
-- Clean, categorize, and enrich completed Texas orders

SELECT
    order_id,
    UPPER(TRIM(customer_name)) AS customer_name_clean,
    city,
    order_status,
    order_amount,
    CASE
        WHEN order_amount >= 500 THEN 'HIGH'
        WHEN order_amount >= 200 THEN 'MEDIUM'
        ELSE 'LOW'
    END AS order_category,
    COALESCE(promo_code, 'NONE') AS promo_code_clean,
    DATEDIFF(ship_date, order_date) AS days_to_ship
FROM orders
WHERE order_status = 'COMPLETED'
  AND state = 'TX';

order_id,customer_name_clean,city,order_status,order_amount,order_category,promo_code_clean,days_to_ship
1001,RAJ KUMAR,Houston,COMPLETED,730.00,HIGH,SAVE10,3
1002,MEENA REDDY,Dallas,COMPLETED,290.00,MEDIUM,NONE,2
1006,RAJ KUMAR,Houston,COMPLETED,575.00,HIGH,SAVE10,3
1008,RAMESH IYER,Austin,COMPLETED,290.00,MEDIUM,FESTIVE20,5
1011,MEENA REDDY,Dallas,COMPLETED,670.00,HIGH,NONE,2
1012,LAKSHMI PILLAI,Houston,COMPLETED,750.00,HIGH,SAVE10,6
1016,SURESH BABU,Austin,COMPLETED,295.00,MEDIUM,WELCOME5,2


### 🟦 PySpark

In [0]:
# Clean, categorize, and enrich completed Texas orders

result_df = (
    orders_df
    .filter(
        (col("order_status") == "COMPLETED") &
        (col("state") == "TX")
    )
    .select(
        "order_id",
        upper(trim(col("customer_name"))).alias("customer_name_clean"),
        "city",
        "order_status",
        "order_amount",
        when(col("order_amount") >= 500, "HIGH")
        .when(col("order_amount") >= 200, "MEDIUM")
        .otherwise("LOW")
        .alias("order_category"),
        coalesce(col("promo_code"), lit("NONE")).alias("promo_code_clean"),
        datediff(col("ship_date"), col("order_date")).alias("days_to_ship")
    )
)

# display data in the DataFrame result_df
display(result_df)

order_id,customer_name_clean,city,order_status,order_amount,order_category,promo_code_clean,days_to_ship
1001,RAJ KUMAR,Houston,COMPLETED,730.00,HIGH,SAVE10,3
1002,MEENA REDDY,Dallas,COMPLETED,290.00,MEDIUM,NONE,2
1006,RAJ KUMAR,Houston,COMPLETED,575.00,HIGH,SAVE10,3
1008,RAMESH IYER,Austin,COMPLETED,290.00,MEDIUM,FESTIVE20,5
1011,MEENA REDDY,Dallas,COMPLETED,670.00,HIGH,NONE,2
1012,LAKSHMI PILLAI,Houston,COMPLETED,750.00,HIGH,SAVE10,6
1016,SURESH BABU,Austin,COMPLETED,295.00,MEDIUM,WELCOME5,2


Same requirement. Same filter → select (with CASE/CAST/COALESCE/date logic) chain. Same result — expressed two ways.

## SECTION 9 — VIEWER CHALLENGE

**PAUSE THE VIDEO AND TRY THIS YOURSELF.**

**Requirement:** For every `RETURNED` order, show the order reference (`ORD-<order_id>`), the cleaned customer name (trimmed and uppercase), and the number of days between order date and ship date.

**Required output:** `order_id`, `order_reference`, `customer_name_clean`, `days_to_ship`

Pause the video for 30–60 seconds. Try it two ways:
1. SQL
2. PySpark

--- PAUSE HERE ---

### 🟨 SQL SOLUTION

In [0]:
%sql
-- Show order reference, cleaned customer name, and days to ship for RETURNED orders

SELECT
    order_id,
    CONCAT('ORD-', CAST(order_id AS STRING)) AS order_reference,
    UPPER(TRIM(customer_name)) AS customer_name_clean,
    DATEDIFF(ship_date, order_date) AS days_to_ship
FROM orders
WHERE order_status = 'RETURNED';

### 🟦 PYSPARK SOLUTION

In [0]:
# Show order reference, cleaned customer name, and days to ship for RETURNED orders

result_df = (
    orders_df
    .filter(col("order_status") == "RETURNED")
    .select(
        "order_id",
        concat(lit("ORD-"), col("order_id").cast("string")).alias("order_reference"),
        upper(trim(col("customer_name"))).alias("customer_name_clean"),
        datediff(col("ship_date"), col("order_date")).alias("days_to_ship")
    )
)

# display data in the DataFrame result_df
display(result_df)

**RESULT:** Both queries return the same 2 orders, with matching `order_reference`, `customer_name_clean`, and `days_to_ship` values.

## Final Check — Did SQL and PySpark Return the Same Result?

We wrote the same business logic in both SQL and PySpark, using the Section 8 combined example.

Now let's verify that they actually returned the same `order_id` values in the same order.

### What will we do?

1. Run the SQL version and store the result in `sql_result`
2. Run the PySpark version and store the result in `pyspark_result`
3. Extract the `order_id` values from each result into Python lists
4. Compare the two lists

Both results are explicitly ordered by `order_id` first, so this is a fair, order-sensitive comparison — we're not relying on any unordered DataFrame having a guaranteed row order.

In [0]:
# Compare SQL and PySpark results to verify that both return the same order_ids in the same order

# Run the SQL query and store the result as a DataFrame
sql_result = spark.sql("""
    SELECT
        order_id,
        UPPER(TRIM(customer_name)) AS customer_name_clean,
        CASE
            WHEN order_amount >= 500 THEN 'HIGH'
            WHEN order_amount >= 200 THEN 'MEDIUM'
            ELSE 'LOW'
        END AS order_category,
        COALESCE(promo_code, 'NONE') AS promo_code_clean,
        DATEDIFF(ship_date, order_date) AS days_to_ship
    FROM orders
    WHERE order_status = 'COMPLETED'
      AND state = 'TX'
    ORDER BY order_id
""")

# Apply the same logic using PySpark and store the result as a DataFrame
pyspark_result = (
    orders_df
    .filter(
        (col("order_status") == "COMPLETED") &
        (col("state") == "TX")
    )
    .select(
        "order_id",
        upper(trim(col("customer_name"))).alias("customer_name_clean"),
        when(col("order_amount") >= 500, "HIGH")
        .when(col("order_amount") >= 200, "MEDIUM")
        .otherwise("LOW")
        .alias("order_category"),
        coalesce(col("promo_code"), lit("NONE")).alias("promo_code_clean"),
        datediff(col("ship_date"), col("order_date")).alias("days_to_ship")
    )
    .orderBy("order_id")
)

# Trigger execution with collect(), bring the result to the driver, and extract order_id values into a Python list
sql_ids = [row["order_id"] for row in sql_result.collect()]

# Trigger execution with collect(), bring the PySpark result to the driver, and extract order_id values into a Python list
pyspark_ids = [row["order_id"] for row in pyspark_result.collect()]

# Display both order_id lists and check whether they match in the same order
print("SQL order_ids:     ", sql_ids)
print("PySpark order_ids: ", pyspark_ids)
print("Same order_ids (order-sensitive):", sql_ids == pyspark_ids)

### Understanding the Result

`.collect()` brings the result rows from Spark to Python — worth doing here only because this is a small, educational dataset.

`row["order_id"]` takes the `order_id` value from each row.

The `[ ... ]` creates a Python list of those values.

So we end up with two lists:

- `sql_ids` → order IDs returned by SQL
- `pyspark_ids` → order IDs returned by PySpark

Finally:

`sql_ids == pyspark_ids`

checks whether both lists contain the same values **in the same order**.

If the result is `True` ✅, our SQL and PySpark versions returned the same ordered `order_id` results.

## SECTION 11 — FINAL CHEAT SHEET

| SQL | PySpark |
|---|---|
| `CASE WHEN ... END` | `when().otherwise()` |
| `CAST(...)` | `.cast()` |
| `IN (...)` | `.isin()` |
| `LIKE` | `.like()` |
| `IS NULL` | `.isNull()` |
| `IS NOT NULL` | `.isNotNull()` |
| `COALESCE` | `coalesce()` |
| `UPPER` | `upper()` |
| `LOWER` | `lower()` |
| `TRIM` | `trim()` |
| `LENGTH` | `length()` |
| `CONCAT` | `concat()` |
| `YEAR` | `year()` |
| `MONTH` | `month()` |
| `DATE_ADD` | `date_add()` |
| `DATE_SUB` | `date_sub()` |
| `DATEDIFF` | `datediff()` |

```
     SQL
      ↕
   PySpark
      ↓
Same Business Logic
```

## Part 2 complete.

**Coming next — Part 3: SQL ↔ PySpark Aggregations & Joins**

- `GROUP BY`
- Aggregate functions
- `JOIN` types
- Window functions (preview)